In [ ]:
!pip install -q -U langgraph langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 3.8 MB/s eta 0:00:00


In [ ]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# 1. Define the Graph State schema
class AgentState(TypedDict):
    user_query: str
    response: str
    is_valid: bool

In [ ]:
# 2. Define Node Functions
def generate_response_node(state: AgentState):
    query = state["user_query"]
    # Simulating LLM output generation
    simulated_answer = f"Processed request: {query.upper()}"
    return {"response": simulated_answer}
def validation_node(state: AgentState):
    # Validates if response meets basic length requirements
    valid = len(state["response"]) > 10
    return {"is_valid": valid}

In [ ]:
#3. Define Routing Logic (Conditional Edge)
def router(state: AgentState):
    if state["is_valid"]:
        return "approved"
    else:
        return "rejected"

In [ ]:
# 4. Build the LangGraph State Machine
builder = StateGraph(AgentState)

# Add Nodes
builder.add_node("generator", generate_response_node)
builder.add_node("validator", validation_node)

# Wire Edges
builder.add_edge(START, "generator")
builder.add_edge("generator", "validator")

# Conditional Routing
builder.add_conditional_edges(
    "validator",
    router,
    {
        "approved": END,
        "rejected": "generator"  # Loops back if validation fails!
    }
)

app = builder.compile()

In [ ]:
# 5. Execute Graph
output = app.invoke({"user_query": "hello langgraph agent"})
print("--- LANGGRAPH EXECUTION COMPLETE ---")
print(f"Final State: {output}")

--- LANGGRAPH EXECUTION COMPLETE ---
Final State: {'user_query': 'hello langgraph agent', 'response': 'Processed request: HELLO LANGGRAPH AGENT', 'is_valid': True}


Standalone MCP Demonstration

In [ ]:
!pip uninstall -y mcp mcp-types
!pip cache purge


Files removed: 8


In [ ]:
!pip install -U mcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.9/357.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 3.5 MB/s eta 0:00:00


In [ ]:
import importlib.metadata

print(importlib.metadata.version("mcp"))

2.1.1


In [ ]:
!pip uninstall -y mcp fastmcp langchain-mcp-adapters langgraph langchain-google-genai
!pip install -q "mcp>=1.0.0,<2.0.0" mcp-types langgraph langchain-google-genai nest_asyncio

Found existing installation: mcp 2.1.1
Uninstalling mcp-2.1.1:
  Successfully uninstalled mcp-2.1.1
Found existing installation: langgraph 1.2.11
Uninstalling langgraph-1.2.11:
  Successfully uninstalled langgraph-1.2.11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.7/224.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 9.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, bu

In [ ]:
import os
import csv
import asyncio
import nest_asyncio
from google.colab import userdata

# LangChain & LangGraph Imports
from langchain_core.tools import StructuredTool
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

nest_asyncio.apply()

# 1. Setup Gemini API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("Riya")

# 2. Initialize Gemini LLM (Use valid model name)
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

# -------------------------------------------------------------
# 3. Local Real-World Tools (MCP Primitive Logic)
# -------------------------------------------------------------
CSV_FILE = "expenses.csv"

def _initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Item", "Amount", "Category"])

def add_expense(item: str, amount: float, category: str) -> str:
    """Logs a new expense with item, amount, and category into CSV."""
    _initialize_csv()
    with open(CSV_FILE, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([item, amount, category])
    return f"Successfully logged expense: {item} - ${amount} ({category})"

def get_expenses() -> str:
    """Retrieves all logged expenses from the CSV file."""
    _initialize_csv()
    with open(CSV_FILE, mode='r') as f:
        reader = csv.reader(f)
        rows = list(reader)
    if len(rows) <= 1:
        return "No expenses recorded yet."
    return "\n".join([", ".join(row) for row in rows])

# -------------------------------------------------------------
# 4. Wrap Tools as LangChain Structured Tools for LangGraph
# -------------------------------------------------------------
mcp_tools = [
    StructuredTool.from_function(
        func=add_expense,
        name="add_expense",
        description="Logs a new expense with item, amount, and category."
    ),
    StructuredTool.from_function(
        func=get_expenses,
        name="get_expenses",
        description="Retrieves all logged expenses from the expense tracker."
    )
]

# -------------------------------------------------------------
# 5. Build & Execute LangGraph Agent
# -------------------------------------------------------------
agent = create_react_agent(llm, mcp_tools)

async def run_agentic_workflow():
    print("--- Task 1: Log an expense ---")
    prompt_1 = "I bought a pizza for $12.50. Category is Food."
    response_1 = await agent.ainvoke({"messages": [("user", prompt_1)]})

    # Correct attribute inspection for LangChain BaseMessage objects
    for msg in response_1["messages"]:
        if msg.type == "ai" and msg.content:
            print(f"\n[Agent Response]: {msg.content}")

    print("\n--- Task 2: Retrieve records ---")
    prompt_2 = "Show me all expenses logged so far."
    response_2 = await agent.ainvoke({"messages": [("user", prompt_2)]})

    for msg in response_2["messages"]:
        if msg.type == "ai" and msg.content:
            print(f"\n[Agent Response]: {msg.content}")

# Run Execution Loop
asyncio.run(run_agentic_workflow())

/tmp/ipykernel_3347/2336888148.py:68: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, mcp_tools)


--- Task 1: Log an expense ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Response]: [{'type': 'text', 'text': "I've logged your expense: **pizza** for **$12.50** under the **Food** category.", 'extras': {'signature': 'ErABCq0BARFNMg9ztrbyNjYo3ae2nyDYDzAl3c10+FY4Bc7sLBRwxW+dIvUmJS5m0SxaMERwLYxieARDPJ2RxgM6GntgfuC4lbvZX+C6f2gvTiCWMPbzmTPJ9WD35pcS1MAyqhFHW3zzFCfAVuyvPVsVG2P61f4nDKcy6RDDaS8ioPBJvcoeYsIwMa3wVTFELDoxV82he/y+LPh1AoCu7dLNfNQtIbzYiCT+K9aAWK79zF4='}}]

--- Task 2: Retrieve records ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Response]: [{'type': 'text', 'text': 'Here are the expenses logged so far:\n\n* **Item:** pizza\n* **Amount:** $12.50\n* **Category:** Food', 'extras': {'signature': 'EoICCv8BARFNMg9+nUbyLuQk0vJ4+SQ3ErHBKJBBbxNIpJ4VJ+8DaPQBSLbgu/5kc4nOgoS/gXTx6qJwe0DDRqqYjafYwS0jYBJCKDM41OwDSNati1i3FhS+NTlUxL4BFg7Slnd9XWNV7STnty0v2sJB9WeRTf/XdTFjPSuBLEamiduV6ebYFRF0t9LS7LMn+OYKU12lnjGkhh5KoGObAhAQ5+ATKVp0emg7ry0rFhos45TKzAM2Uc0UmVA/Ff04spgV9zt460XeaK7ap1ZzRfHi3/pignIcZmQ5TrxxgsDQeBIC7WdEjX3lwgbfT+ZmJ7L1nJIdosaFzo7xjcHOqiEMvA4h'}}]


In [ ]:
import os
import csv
import asyncio
import nest_asyncio
from google.colab import userdata

from langchain_core.tools import StructuredTool
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

nest_asyncio.apply()

# 1. API & Model Setup
os.environ["GOOGLE_API_KEY"] = userdata.get("Riya")
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

# 2. Expense Tools
CSV_FILE = "expenses.csv"

def _initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode='w', newline='') as f:
            csv.writer(f).writerow(["Item", "Amount", "Category"])

def add_expense(item: str, amount: float, category: str) -> str:
    _initialize_csv()
    with open(CSV_FILE, mode='a', newline='') as f:
        csv.writer(f).writerow([item, amount, category])
    return f"Logged: {item} - ${amount} ({category})"

def get_expenses() -> str:
    _initialize_csv()
    with open(CSV_FILE, mode='r') as f:
        rows = list(csv.reader(f))
    return "No expenses." if len(rows) <= 1 else "\n".join([", ".join(r) for r in rows[1:]])

# 3. Wrap Tools & Build Agent
mcp_tools = [
    StructuredTool.from_function(func=add_expense, name="add_expense", description="Logs an expense."),
    StructuredTool.from_function(func=get_expenses, name="get_expenses", description="Gets logged expenses.")
]

agent = create_agent(llm, mcp_tools)

# 4. Helper to extract raw text string
def get_clean_text(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join([b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text"])
    return str(content)

# 5. Execution
async def run_agentic_workflow():
    res1 = await agent.ainvoke({"messages": [("user", "I bought a pizza for $12.50. Category is Food.")]})
    print(get_clean_text(res1["messages"][-1].content))

    res2 = await agent.ainvoke({"messages": [("user", "Show me all expenses logged so far.")]})
    print(get_clean_text(res2["messages"][-1].content))

asyncio.run(run_agentic_workflow())

I've logged your expense: **pizza** for **$12.50** under the **Food** category.
Here are the expenses logged so far:

1. **Item:** Pizza | **Amount:** $12.50 | **Category:** Food
2. **Item:** Pizza | **Amount:** $12.50 | **Category:** Food
